In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use one T4 only

!pip uninstall -y -q torchao
!pip install -q trl peft wandb

import torch, transformers, trl, peft
print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| peft", peft.__version__)

GPU: Tesla T4
torch 2.11.0+cu130 | transformers 5.20.2 | trl 1.18.2 | peft 0.21.2


In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_ENTITY"] = "csun-self-improving-llm"
os.environ["WANDB_PROJECT"] = "self-improving-llm"
print("Settings ready for", os.environ["WANDB_ENTITY"], "/", os.environ["WANDB_PROJECT"])

Settings ready for csun-self-improving-llm / self-improving-llm


In [4]:
import re, random
from datasets import load_dataset, Dataset

NUMBER = re.compile(r"-?\d+(?:,\d{3})*(?:\.\d+)?")

def last_number(text):
    found = NUMBER.findall(text)
    return found[-1].replace(",", "") if found else None

raw = load_dataset("microsoft/orca-math-word-problems-200k", split="train[:2000]")

rows = []
for item in raw:
    ref = last_number(item["answer"])
    if ref is not None:
        rows.append({
            "prompt": [{"role": "user", "content":
                        item["question"] + "\n\nShow your steps, then end with 'Answer: <number>'."}],
            "answer": ref,
        })

random.seed(42)
dataset = Dataset.from_list(random.sample(rows, 50))
print(dataset[0]["prompt"][0]["content"])
print("Reference answer:", dataset[0]["answer"])

README.md:   0%|          | 0.00/6.91k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 84.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/200035 [00:00<?, ? examples/s]

18 is the result of multiplying a number by 2. What do you get when you subtract 4 from this number?

Show your steps, then end with 'Answer: <number>'.
Reference answer: 5


In [5]:
def temp_correctness_reward(completions, answer, **kwargs):
    """TEMPORARY: 1.0 if the number after 'Answer:' matches the reference, else 0.0.
    Replaced by the team's real checker during integration."""
    rewards = []
    for completion, ref in zip(completions, answer):
        text = completion[0]["content"]
        answer_lines = re.findall(r"Answer:\s*([^\n]+)", text)
        predicted = last_number(answer_lines[-1]) if answer_lines else None
        try:
            correct = predicted is not None and abs(float(predicted) - float(ref)) < 1e-6
        except ValueError:
            correct = False
        rewards.append(1.0 if correct else 0.0)
    return rewards

fake = [[{"content": "23 - 9 = 14. Answer: 14."}], [{"content": "Answer: 15"}], [{"content": "no answer"}]]
print(temp_correctness_reward(fake, ["14", "14", "14"]))   # expect [1.0, 0.0, 0.0]

[1.0, 0.0, 0.0]


In [6]:
import wandb
wandb.finish()   # close the first run, so this one is logged separately

import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules="all-linear", task_type="CAUSAL_LM")

config = GRPOConfig(
    output_dir="smoke-test",
    run_name="smoke-orca-0.5b-len512",
    max_steps=20,
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,   # 16 answers per step = 2 problems x 8 attempts
    num_generations=8,               # attempts per problem
    max_completion_length=512,       # answer-length cap (was 256: 64% of answers got cut off)
    logging_steps=1,
    report_to="wandb",
    fp16=True,
    bf16=False,                      # T4 GPUs don't support bf16
    save_strategy="no",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=temp_correctness_reward,
    args=config,
    train_dataset=dataset,
    peft_config=lora,
)

start = time.time()
trainer.train()
print(f"Training took {(time.time() - start) / 60:.1f} minutes")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: aartigapo (csun-self-improving-llm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261121_035048-ck408rnd
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run smoke-orca-0.5b-len512
wandb: ⭐️ View project at https://wandb.ai/csun-self-improving-llm/self-improving-llm
wandb: 🚀 View run at https://wandb.ai/csun-self-improving-llm/self-improving-llm/runs/ck408rnd


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.063368
5,0.000000
6,-0.022304
7,0.009906
8,0.003312
9,0.016904
10,-0.002223


Training took 11.1 minutes


In [7]:
import pandas as pd

logs = pd.DataFrame([row for row in trainer.state.log_history if "reward" in row])

wanted = ["step", "reward", "frac_reward_zero_std", "completions/clipped_ratio",
          "completions/mean_length", "num_tokens", "step_time"]
cols = [c for c in wanted if c in logs.columns]

pd.set_option("display.width", 200)
print(logs[cols].round(3).to_string(index=False))
print("\nAverages over all 20 steps:")
print(logs[cols[1:]].mean().round(3).to_string())

 step  reward  frac_reward_zero_std  completions/clipped_ratio  completions/mean_length  num_tokens  step_time
    1   0.000                   1.0                      0.062                  392.250      7380.0     40.586
    2   0.000                   1.0                      0.188                  401.875     15298.0     34.627
    3   0.000                   1.0                      0.625                  451.250     23854.0     35.256
    4   0.188                   0.0                      0.062                  241.875     28940.0     33.775
    5   0.000                   1.0                      0.000                  278.438     34611.0     31.438
    6   0.125                   0.5                      0.125                  261.750     40063.0     33.707
    7   0.125                   0.5                      0.188                  310.000     46335.0     33.893
    8   0.125                   0.0                      0.062                  246.812     51444.0     33.636
 